In [2]:
import pandas as pd
from pathlib import Path

base = Path(r'c:/Users/MSUSERSL123/Documents/maersk/supply-chain-platform')
df = pd.read_csv(base / 'data' / 'raw' / 'shipments.csv')
print(df.shape)
print(df['status'].value_counts(dropna=False).head(20).to_dict())
print('missing_status=', int(df['status'].isna().sum()))
status_values = df['status'].fillna('').astype(str).str.strip().str.upper()
valid_statuses = {'DELAYED','LATE','LATE_ARRIVAL','ARRIVAL_DELAYED','DELAY','DELIVERED','ON_TIME','ON-TIME','ON TIME','IN_TIME','ARRIVED','ON-TIME-ARRIVAL'}
filtered = df[status_values.isin(valid_statuses)]
print('filtered_rows=', len(filtered))
print(filtered['status'].value_counts().to_dict())
from src.ml.features import build_delay_target
labels = build_delay_target(filtered)
print('target_counts=', labels.dropna().value_counts().to_dict())
print('nan_rows=', int(labels.isna().sum()))

(5028, 14)
{'DELIVERED': 2778, 'DELAYED': 1901, 'CANCELLED': 203, nan: 54, 'COMPLETED': 34, 'Complete': 30, 'delivered': 28}
missing_status= 54
filtered_rows= 4707
{'DELIVERED': 2778, 'DELAYED': 1901, 'delivered': 28}
target_counts= {np.int64(0): 2806, np.int64(1): 1901}
nan_rows= 0


In [4]:
from src.ml.train import train_model

metrics = train_model(db_path='data/raw/shipments.csv', out_dir='models')
print(metrics)


ValueError: The split labels still contain missing values after target construction. Please clean or reclassify the shipments before training.